# Tourism Package Prediction — Colab Runner

This notebook clones the project's GitHub repository and runs the full MLOps pipeline (data registration -> data preparation -> model training with MLflow tracking) live, then loads the trained model and makes a sample prediction so you can see it working end to end.

- **Repository:** https://github.com/niteshtripa/tourism-package-prediction
- **Live app (already deployed):** https://tourism-package-prediction-gqebjzdhhzrn33gpitu3mw.streamlit.app/

**How to run:** Runtime -> Run all. Takes about 2-3 minutes.

## 1. Install dependencies

In [ ]:
!pip install -q mlflow==3.0.1 xgboost==2.1.1

## 2. Clone the project repository

In [ ]:
!git clone https://github.com/niteshtripa/tourism-package-prediction.git
%cd tourism-package-prediction

## 3. Data Registration

Validates that `tourism.csv` has the expected columns and prints a summary.

In [ ]:
!python tourism_project/model_building/data_register.py

## 4. Data Preparation

Cleans the data and creates the train/test split.

In [ ]:
!python tourism_project/model_building/prep.py

## 5. Model Training with MLflow Tracking

Tunes an XGBoost classifier with GridSearchCV, logs the run to MLflow, evaluates it, and saves the best model.

In [ ]:
!python tourism_project/model_building/train.py

### Inspect the MLflow experiment runs

In [ ]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
runs = mlflow.search_runs(experiment_names=["tourism-package-prediction"])
runs[[c for c in runs.columns if c.startswith("metrics.") or c.startswith("params.")]]

## 6. See a Live Prediction

Loads the model that was just trained and saved to `tourism_project/deployment/model.joblib`, and runs it on a sample customer to show the actual output — this is the same model the Streamlit app uses.

In [ ]:
import joblib
import pandas as pd

model = joblib.load("tourism_project/deployment/model.joblib")

sample_customer = pd.DataFrame([{
    "Age": 37,
    "TypeofContact": "Self Enquiry",
    "CityTier": 1,
    "DurationOfPitch": 15,
    "Occupation": "Salaried",
    "Gender": "Female",
    "NumberOfPersonVisiting": 3,
    "NumberOfFollowups": 4,
    "ProductPitched": "Deluxe",
    "PreferredPropertyStar": 3.0,
    "MaritalStatus": "Single",
    "NumberOfTrips": 3,
    "Passport": 1,
    "PitchSatisfactionScore": 3,
    "OwnCar": 1,
    "NumberOfChildrenVisiting": 1,
    "Designation": "Manager",
    "MonthlyIncome": 23000,
}])

prediction = model.predict(sample_customer)[0]
probability = model.predict_proba(sample_customer)[0, 1]

print("Prediction:", "Will purchase" if prediction == 1 else "Will not purchase")
print(f"Probability of purchase: {probability:.1%}")
sample_customer

### Try your own customer

Edit any value below and re-run this cell to see how the prediction changes.

In [ ]:
my_customer = pd.DataFrame([{
    "Age": 45,
    "TypeofContact": "Company Invited",
    "CityTier": 2,
    "DurationOfPitch": 20,
    "Occupation": "Small Business",
    "Gender": "Male",
    "NumberOfPersonVisiting": 4,
    "NumberOfFollowups": 5,
    "ProductPitched": "Super Deluxe",
    "PreferredPropertyStar": 5.0,
    "MaritalStatus": "Married",
    "NumberOfTrips": 2,
    "Passport": 1,
    "PitchSatisfactionScore": 4,
    "OwnCar": 1,
    "NumberOfChildrenVisiting": 2,
    "Designation": "Senior Manager",
    "MonthlyIncome": 35000,
}])

prediction = model.predict(my_customer)[0]
probability = model.predict_proba(my_customer)[0, 1]
print("Prediction:", "Will purchase" if prediction == 1 else "Will not purchase")
print(f"Probability of purchase: {probability:.1%}")

## 7. The Full Interactive App

The form-based web app (same model, point-and-click interface) is already deployed and live — Streamlit apps need their own web server, so they can't run directly inside a Colab cell. Open it here:

### 🔗 https://tourism-package-prediction-gqebjzdhhzrn33gpitu3mw.streamlit.app/

## 8. GitHub Actions Pipeline

This entire sequence (steps 3-5 above) also runs automatically on GitHub every time code is pushed to `main` — see it here:

### 🔗 https://github.com/niteshtripa/tourism-package-prediction/actions